# s4figure.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 按新数据的实际时间长度生成月度坐标
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import font_manager
from pathlib import Path
import cartopy.mpl.ticker as cticker

# 注册项目内 Times New Roman Regular/Bold，避免字体回退
times_font_dir = Path(
    configured_path('font_dir')
)
times_font_files = [
    times_font_dir / 'Times New Roman.ttf',
    times_font_dir / 'Times New Roman Bold.ttf'
]
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(
    font_manager.findfont('Times New Roman', fallback_to_default=False)
).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold'
})

# Pred 和 Obs 使用相同海洋掩码以及相同时间、空间坐标
mld_pred = ds['MLD_pred'].where(mask_ocean)
mld_obs = ds['MLD_true'].where(mask_ocean)

def month_of_maximum(da):
    """计算每个网格点序列最大值出现的月份（1--12）。"""
    valid = da.notnull().any('time')
    max_index = da.fillna(-np.inf).argmax('time').compute()
    return da['time'].dt.month.isel(time=max_index).where(valid).astype(float)

def wrap_longitude_30_390(field):
    """将经度重排到 [30, 390)，并添加循环列消除接缝。"""
    lon_name = 'lon' if 'lon' in field.coords else 'longitude'
    wrapped_lon = ((field[lon_name] - 30) % 360) + 30
    wrapped = field.assign_coords({lon_name: wrapped_lon}).sortby(lon_name)
    cyclic = wrapped.isel({lon_name: 0}).assign_coords(
        {lon_name: wrapped[lon_name].isel({lon_name: 0}) + 360}
    )
    return xr.concat([wrapped, cyclic], dim=lon_name)

pred_max_month = wrap_longitude_30_390(month_of_maximum(mld_pred))
obs_max_month = wrap_longitude_30_390(month_of_maximum(mld_obs))

# 指定的 12 个离散颜色依次对应 1--12 月
rgb_colors = np.array([
    [17, 137, 216],
    [19, 166, 207],
    [22, 160, 212],
    [23, 183, 178],
    [26, 193, 139],
    [199, 189, 86],
    [247, 182, 72],
    [254, 186, 42],
    [255, 209, 10],
    [255, 248, 7]
], dtype=np.float32)
deep_blue = np.array([0, 50, 150], dtype=np.float32)
interp_ratios = np.linspace(0, 1, 4)[1:-1]
new_colors = np.array([
    deep_blue * (1 - ratio) + rgb_colors[0] * ratio
    for ratio in interp_ratios
])
colors_12 = np.vstack([new_colors, rgb_colors]) / 255.0
month_cmap = ListedColormap(colors_12, name='month_of_max')
month_norm = BoundaryNorm(np.arange(0.5, 13.5, 1), 12)
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

projection = ccrs.Robinson(central_longitude=210)
data_crs = ccrs.PlateCarree()
fig, axes = plt.subplots(
    1, 2, figsize=(14.5, 5.8), subplot_kw={'projection': projection},
    constrained_layout=False
)
fig.subplots_adjust(left=0.015, right=0.995, bottom=0.16, top=0.94, wspace=0.035)

panels = [
    (pred_max_month, '(a) Month of Maximum MLD (SJTU-ViT)'),
    (obs_max_month, '(b) Month of Maximum MLD (Obs)')
]

for k, (ax, (field, title)) in enumerate(zip(axes, panels)):
    mesh = field.plot.pcolormesh(
        ax=ax, transform=data_crs, cmap=month_cmap, norm=month_norm,
        add_colorbar=False, add_labels=False, rasterized=True, shading='auto'
    )
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='0.30', edgecolor='none', zorder=3)
    ax.coastlines(linewidth=0.45, color='0.25', zorder=4)
    ax.set_title(title, fontsize=20, pad=8)

    label_sides = ['left', 'bottom', 'x', 'y'] if k == 0 else ['bottom', 'x']
    gl = ax.gridlines(
        crs=data_crs, draw_labels=label_sides, linewidth=0.35, color='0.5',
        alpha=0.45, linestyle='--', x_inline=False, y_inline=False
    )
    # 地图从 30°E 开始；标签使用标准东/西经表达
    gl.xlocator = ticker.FixedLocator([-150, -90, -30, 30, 90, 150])
    gl.ylocator = ticker.FixedLocator([-80, -40, 0, 40, 80])
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16, 'family': 'serif'}
    gl.ylabel_style = {'size': 16, 'family': 'serif'}

    cbar = fig.colorbar(
        mesh, ax=ax, location='bottom', orientation='horizontal',
        ticks=np.arange(1, 13), shrink=0.85, pad=0.08,
        fraction=0.055, aspect=35
    )
    cbar.set_ticklabels(month_labels)
    cbar.ax.tick_params(labelsize=14, length=5)

output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 's4figure.png'
fig.savefig(output_path, dpi=600, bbox_inches='tight', pad_inches=0, facecolor='white')
print(f'Figure saved to: {output_path}')
plt.show()